# IL-only FULL GPU step (forward + backward + AdamW, no C++/torch)

3 cells. The Python reference lives in the repo: `Release/cuda_backend/examples/ref_full_gpu_step.py`

Expected: 5 decreasing `LOSS` lines, `loss maxerr` ~1e-7, `W1/W2 step1 maxerr` ~1e-7, `KET LUAN: OK`.


In [ ]:
# 1/3: install .NET 10 (no mono, no C/C++ build)
import os, subprocess
r=subprocess.run('wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh && bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /kaggle/working/dotnet', shell=True, capture_output=True, text=True, timeout=1200)
print('dotnet install rc=', r.returncode, (r.stderr or '')[-300:])
v=subprocess.run('/kaggle/working/dotnet/dotnet --version', shell=True, capture_output=True, text=True)
print('dotnet', (v.stdout or '').split('\n')[0])


In [ ]:
# 2/3: clone backend + symlink CUDA libs as .dll (P/Invoke names)
import os, subprocess, glob
d='/kaggle/working/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('CUDA backend HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
libdir='/kaggle/working/tkvlib'
os.makedirs(libdir, exist_ok=True)
def pick(pats):
    for pat in pats:
        f=[p for p in glob.glob(pat) if os.path.exists(os.path.realpath(p))]
        if f: return sorted(f)[0]
    return None
libs={
 'cudart.dll': pick(['/usr/local/cuda*/lib64/libcudart.so','/usr/lib/x86_64-linux-gnu/libcudart.so*']),
 'cublas.dll': pick(['/usr/local/cuda*/lib64/libcublas.so','/usr/lib/x86_64-linux-gnu/libcublas.so*']),
 'cuda.dll': pick(['/usr/lib/x86_64-linux-gnu/libcuda.so*','/usr/lib64/libcuda.so*','/usr/local/nvidia/lib64/libcuda.so*','/usr/local/nvidia/lib/libcuda.so*','/usr/local/cuda/compat/libcuda.so*'])
}
for nm,tgt in libs.items():
    print(nm,'->',tgt)
    if tgt:
        p=libdir+'/'+nm
        if os.path.lexists(p): os.remove(p)
        os.symlink(tgt,p)
os.environ['LD_LIBRARY_PATH']=libdir+':'+os.environ.get('LD_LIBRARY_PATH','')
print('LD_LIBRARY_PATH ok')


In [ ]:
# 3/3: run full GPU step under .NET 10, then compare with the repo reference
import subprocess, os, json
exe=d+'/Release/cuda_backend/examples/ilgpu_step_full_gpu.exe'
print('exe bytes=', os.path.getsize(exe) if os.path.exists(exe) else 0)
cfg={'runtimeOptions':{'tfm':'net10.0','framework':{'name':'Microsoft.NETCore.App','version':'10.0.0'}}}
open(os.path.dirname(exe)+'/ilgpu_step_full_gpu.runtimeconfig.json','w').write(json.dumps(cfg))
r=subprocess.run(['/kaggle/working/dotnet/dotnet', exe], env=dict(os.environ), capture_output=True, text=True, timeout=600)
print('dotnet rc=', r.returncode)
print(r.stdout[:3000])
print('STDERR:', (r.stderr or '')[:1200])
exec(open(d+'/Release/cuda_backend/examples/ref_full_gpu_step.py').read())
